# 6장 · 정규분포 🔔

웹 챕터 **"정규분포 (핵심)"** 과 짝을 이루는 Python 실습입니다.
확률 = 곡선 아래 넓이, 68-95-99.7 규칙, z 점수를 `scipy.stats.norm` 으로 직접 계산해 봅니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch6')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 한글 폰트 (Windows) — 이게 없으면 그래프의 한글이 깨져요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

rng = np.random.default_rng(42)  # 시드 고정 — 실행할 때마다 같은 결과

## 1. 밀도곡선: 확률은 '넓이'

연속 확률변수는 '정확히 한 값'의 확률이 0 이라, **구간의 확률 = 곡선 아래 넓이**로 말합니다.
(넓이 = 잘게 쪼개서 다 더한 것, 곧 적분 ∫ 의 직관. 계산은 scipy 의 `cdf` 가 해줍니다.)

성인 키를 N(170, 5²) 모형으로 그려봅시다.

In [ ]:
mu, sigma = 170, 5
X = stats.norm(mu, sigma)

xs = np.linspace(150, 190, 400)
ys = X.pdf(xs)  # pdf = 확률밀도(곡선의 높이)

plt.figure(figsize=(7, 3.5))
plt.plot(xs, ys, color='#6366f1', lw=2)
sel = (xs >= 165) & (xs <= 175)
plt.fill_between(xs[sel], ys[sel], color='#6366f1', alpha=0.3)
plt.title('N(170, 5²): 색칠한 넓이 = P(165 ≤ X ≤ 175)')
plt.xlabel('키 (cm)')
plt.ylabel('밀도')
plt.show()

# cdf(x) = 'x 이하일 확률(누적 넓이)' → 구간 넓이는 큰 쪽 - 작은 쪽
area = X.cdf(175) - X.cdf(165)
print('P(165 ≤ X ≤ 175) =', round(area, 4), ' → 약 68% (μ±1σ 구간!)')

## 2. 68-95-99.7 규칙 검증

표준정규분포 N(0, 1) 에서 μ±1σ, ±2σ, ±3σ 안의 넓이를 직접 계산합니다.

In [ ]:
Z = stats.norm(0, 1)  # 표준정규분포
for k in [1, 2, 3]:
    area = Z.cdf(k) - Z.cdf(-k)
    print(f'μ±{k}σ 안의 비율: {area * 100:.1f}%')

## 3. z 점수: 표준화

$$z = \frac{x - \mu}{\sigma}$$

웹 챕터의 StepQuiz 예제: 키 180cm (μ=170, σ=5) → z = 2 → 상위 약 2.5%.

In [ ]:
x = 180
z = (x - mu) / sigma
print('z 점수 =', z)
print('상위 비율 =', round((1 - Z.cdf(z)) * 100, 2), '%   (95% 규칙: 바깥 5%의 절반 ≈ 2.5%)')

# 연습문제 검산 — IQ N(100, 15²)에서 상위 16%의 경계
print('IQ 상위 15.9% 경계 =', round(stats.norm(100, 15).ppf(1 - 0.159), 1), ' (≈ μ+1σ = 115)')

## 4. 정규 난수: 히스토그램 위에 밀도곡선

정규분포에서 뽑은 난수 1만 개의 히스토그램이 이론 곡선에 달라붙는 걸 확인합니다.
(AI 의 가중치 초기화가 바로 이 '정규분포에서 난수 뽑기'예요.)

In [ ]:
samples = rng.normal(mu, sigma, size=10_000)

plt.figure(figsize=(7, 3.5))
plt.hist(samples, bins=40, density=True, color='#a5b4fc', edgecolor='white', label='난수 10,000개')
plt.plot(xs, ys, color='#ef4444', lw=2, label='이론 밀도곡선')
plt.title('정규 난수: 히스토그램이 곡선을 따라간다')
plt.xlabel('값')
plt.legend()
plt.show()

print('난수의 평균 =', round(samples.mean(), 2), ' (이론값 170)')
print('난수의 표준편차 =', round(samples.std(), 2), ' (이론값 5)')

## 5. 갈톤 보드: 우연이 쌓이면 종 모양

공이 핀에 부딪혀 왼쪽/오른쪽 반반 확률로 튀며 10줄을 내려오면,
'오른쪽으로 간 횟수'는 이항분포 B(10, 0.5)를 따르고, 쌓인 모양이 종 모양이 됩니다.
(작은 우연의 합 → 정규분포. 7장 중심극한정리의 복선!)

In [ ]:
# 공 5000개, 핀 10줄: 각 줄에서 0(왼쪽) 또는 1(오른쪽)
balls = rng.integers(0, 2, size=(5000, 10)).sum(axis=1)

k = np.arange(11)
plt.figure(figsize=(7, 3.5))
plt.hist(balls, bins=np.arange(-0.5, 11.5, 1), density=True,
         color='#34d399', edgecolor='white', label='갈톤 보드 시뮬레이션')
plt.plot(k, stats.binom.pmf(k, 10, 0.5), 'o--', color='#ef4444', label='이항분포 B(10, 0.5)')
plt.title('갈톤 보드: 공 5000개가 쌓인 모양')
plt.xlabel('오른쪽으로 튄 횟수')
plt.legend()
plt.show()

## 정리

- 연속 확률변수: 확률 = **밀도곡선 아래 넓이** (전체 넓이 1), `cdf` 로 계산
- **정규분포 N(μ, σ²)**: μ 가 중심, σ 가 폭인 종 모양
- **68-95-99.7**: μ±1σ / ±2σ / ±3σ 안에 약 68% / 95% / 99.7%
- **z = (x−μ)/σ**: 어떤 정규분포든 표준정규분포 N(0,1) 로 변환 → "상위 몇 %" 즉시 계산
- AI 연결: 가중치 초기화·디퓨전 노이즈·오차 모델링이 모두 정규분포에서 출발

다음 노트북: `ch07_lln_clt.ipynb`